# **Gender Pronouns — 2nd Person Extraction**<br>

## **1.** OpenSub Dataset Upload

In [ ]:
import sys, re, pickle, os
import pandas as pd
sys.path.append('../modules')

In [ ]:
with open("../local_data/gender_pronouns/1st_person/data_final_1st_person.pkl", 'rb') as f:
    data_1st_person = pickle.load(f)

df_female = data_1st_person[data_1st_person['self_ref'] == 'F'][['eng_text', 'pol_text']].reset_index(drop=True)
df_male = data_1st_person[data_1st_person['self_ref'] == 'M'][['eng_text', 'pol_text']].reset_index(drop=True)
df_first_person = data_1st_person[data_1st_person['self_ref'] == 'NA'][['eng_text', 'pol_text']].reset_index(drop=True)
df_text = data_1st_person[data_1st_person['self_ref'] == 'NA_OTHER'][['eng_text', 'pol_text']].reset_index(drop=True)

## **2.** Helper Functions

In [ ]:
def extract_sample(df_main, mask_2nd, name_df):
    print(f"[{name_df}]: {(mask_2nd.sum()/len(df_main)*100):.2f}% of examples extracted")
    df_mask = df_main[mask_2nd].reset_index(drop=True)
    return df_mask, df_main[~mask_2nd].reset_index(drop=True)

def search(snt, re_pattern):
    return bool(re.search(re_pattern, snt, re.IGNORECASE))

def apply_mask(df, re_pattern, col='pol_text'):
    return df[col].apply(lambda snt: search(snt, re_pattern))

In [ ]:
def report_counts(**counts):
    print(" | ".join(f"{k.replace('_', ' ')}: {v.sum() if hasattr(v, 'sum') else v}" for k, v in counts.items()))

def report_shapes(**dfs):
    print(" | ".join(f"{k}: {v.shape[0]}" for k, v in dfs.items()))

## **3.** Second-Person Sentence Filter

In [ ]:
re_2ndperson = r"\b(?:you['’](?:re|ve|ll|d)|you|your|yours|yourself|thou|thee|thy|thine|thyself)\b"

mask_2nd_person = df_text['eng_text'].str.contains(re_2ndperson, case=False, na=False, regex=True)
mask_2nd_person_1st_na = df_first_person['eng_text'].str.contains(re_2ndperson, case=False, na=False, regex=True)
mask_2nd_person_1st_m = df_male['eng_text'].str.contains(re_2ndperson, case=False, na=False, regex=True)
mask_2nd_person_1st_f = df_female['eng_text'].str.contains(re_2ndperson, case=False, na=False, regex=True)

print(f"Num examples >>\n[df_text]:{mask_2nd_person.sum():>17}\n[df_first_person]:   {mask_2nd_person_1st_na.sum()}\n\
[df_male]: {mask_2nd_person_1st_m.sum():>16}\n[df_female]: {mask_2nd_person_1st_f.sum():>14}")

In [ ]:
df_2nd_person, df_text = extract_sample(df_text, mask_2nd_person, 'df_text')
df_2nd_person_na, df_first_person = extract_sample(df_first_person, mask_2nd_person_1st_na, 'df_first_person')
df_2nd_person_m, df_male = extract_sample(df_male, mask_2nd_person_1st_m, 'df_male')
df_2nd_person_f, df_female = extract_sample(df_female, mask_2nd_person_1st_f, 'df_female')

In [ ]:
df_2nd_person_f['self_ref'] = 'F'
df_2nd_person_m['self_ref'] = 'M'
df_2nd_person_na['self_ref'] = 'NA'
df_2nd_person['self_ref'] = 'NA_OTHER'

df_2nd_person = pd.concat([df_2nd_person, df_2nd_person_m, df_2nd_person_f, df_2nd_person_na], ignore_index=True)

## **4.** Second-Person Pronouns Extraction

In [ ]:
df_proc_f = pd.DataFrame(columns=df_2nd_person.columns)
df_proc_m = pd.DataFrame(columns=df_2nd_person.columns)
df_proc_p = pd.DataFrame(columns=df_2nd_person.columns)

### **GROUP 0**: First-Person Sentences Final Classification

In [ ]:
edge_pat_m = r"\b((od|wy|u|\b)dzia|(m|b|skrz)yd|(ob|\b)strza|(syg|kardy|ka)na|(admi|gene)ra|(po|u)mys|[tp]y|(m|tyt)u|(an|kośc)io|\
anio|diab|(ma|o|krze)s|dzie|peda|zespo|kryszta|świat|(gar|źró|sio)d|(ko|\b)z|zapa|(mater|c)ia|(kow|zwierci|popych)ad|(\b|apo)sto|(proto|\b)ko)łem\b"

mask_refs = (df_2nd_person['self_ref'] != 'M') & (df_2nd_person['self_ref'] != 'F')

edge_case_m = apply_mask(df_2nd_person, edge_pat_m)
mask_m = (apply_mask(df_2nd_person, r"\b\w+(łem|[łl]bym)\b") & mask_refs) & ~edge_case_m

edge_case_f = apply_mask(df_2nd_person, r"\b([wzk]|(po|wy|prze)sy|(za|od|po|wy)wo|dzia|po|zdo)łam\b")
mask_f = (apply_mask(df_2nd_person, r"\b\w+(łam|[łl]abym)\b") & mask_refs) & ~edge_case_f

print(f"Num edge cases --> [Female]: {edge_case_f.sum()} | [Male]: {edge_case_m.sum()}")
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()}")

df_2nd_person.loc[mask_m, 'self_ref'] = 'M'
df_2nd_person.loc[mask_f, 'self_ref'] = 'F'

### **GROUP 1A**: Past tense 2nd person (-aś / -eś / -ście)

In [ ]:
def extract_pronouns(df_main, df_proc_f, df_proc_m, df_proc_p, mask_f, mask_m, mask_p):
    df_proc_f = pd.concat([df_proc_f, df_main[mask_f]], ignore_index=True)
    df_proc_m = pd.concat([df_proc_m, df_main[mask_m]], ignore_index=True)
    
    if df_proc_p is not None:
        df_proc_p = pd.concat([df_proc_p, df_main[mask_p]], ignore_index=True)
        df_main = df_main[~(mask_f | mask_m | mask_p)].reset_index(drop=True)
        return df_main, df_proc_f, df_proc_m, df_proc_p
    else:
        df_main = df_main[~(mask_f | mask_m)].reset_index(drop=True)
        return df_main, df_proc_f, df_proc_m

In [ ]:
edge_case_m = apply_mask(df_2nd_person, r"\b((prz[ye]|(p|\b)od|[zn]a|\b|w[yz]|[zwu]|roz|po)[nw]i|\w*(gdzi|jaki|[żr])|[ia]l|jest|czyj)eś\b")
edge_case_f = apply_mask(df_2nd_person, r"\b(.|\w*g|chyb|któr|ad)aś\b") | apply_mask(df_2nd_person, r"\bjakaś\b(?!.*\bty\b)")
edge_case_p = apply_mask(df_2nd_person, r"\b([mgt]o|(w|z|pod)ej|(jede|osiem|czter|kilka)na|(\b|nie)szczę|(o|rze)czywi|gu|gu\
                                        |(wy|do|u)j|(c|m|dw)ie|(zaje|oso)bi|gu|gu|li|oszu|nare|zem|uroczy|(kon|\b)tek)ście\b")

mask_m = apply_mask(df_2nd_person, r"\b\w+eś\b") & ~edge_case_m
mask_f = apply_mask(df_2nd_person, r"\b\w+aś\b") & ~edge_case_f
mask_p = apply_mask(df_2nd_person, r"\b\w+ście\b") & ~edge_case_p

print(f"Num edge cases --> [Female]: {edge_case_f.sum()} | [Male]: {edge_case_m.sum()} | [Plural]: {edge_case_p.sum()}")
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()} | [Plural]: {mask_p.sum()}")

df_2nd_person, df_proc_f, df_proc_m, df_proc_p = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, df_proc_p, mask_f, mask_m, mask_p)

### **GROUP 1B**: Past tense 2nd person **( -łabyś / -łbyś / -byście )**

In [ ]:
mask_m = apply_mask(df_2nd_person, r"\b\w+[łl]by[śs]\b")
mask_f = apply_mask(df_2nd_person, r"\b\w+[łl]aby[śs]\b")
mask_p = apply_mask(df_2nd_person, r"\b\w*by[śs]cie\b")
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()} | [Plural]: {mask_p.sum()} (Previously extracted)")
df_2nd_person, df_proc_f, df_proc_m, df_proc_p = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, df_proc_p, mask_f, mask_m, mask_p)

### **GROUP 2A**: Formal address **( pan / pani / pa-{nie/nowie} )**

In [ ]:
edge_case_m = apply_mask(df_2nd_person, r"\b((tw|m)ój|[wn]asz|on|(\b|tam)ten)\b\s+\bpan\b")
edge_case_f = apply_mask(df_2nd_person, r"\b((tw|m)oj|[wn]asz|on|(\b|tam)t)a\b\s+\bpani\b")

mask_m = apply_mask(df_2nd_person, r"\bpan\b") & ~edge_case_m
mask_f = apply_mask(df_2nd_person, r"\bpani\b") & ~edge_case_f

mask_p = apply_mask(df_2nd_person, r"\bladies\b", 'eng_text') & apply_mask(df_2nd_person, r"\bpanie\b")
mask_p = mask_p | apply_mask(df_2nd_person, r"\b((tw|m)oje|drogie)\b\s+\bpanie\b") | apply_mask(df_2nd_person, r"\bpanowie\b")

print(f"Num edge cases --> [Female]: {edge_case_f.sum()} | [Male]: {edge_case_m.sum()}")
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()} | [Plural]: {mask_p.sum()}")
df_2nd_person, df_proc_f, df_proc_m, df_proc_p = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, df_proc_p, mask_f, mask_m, mask_p)

### **GROUP 2B**: Formal address **( pana / panią / pa-{niom/nom} )**

In [ ]:
edge_case_m = apply_mask(df_2nd_person, r"\b((tw|m)ojego|[wn]aszego|go|(\b|tam)tego)\b\s+\bpana\b")
edge_case_f = apply_mask(df_2nd_person, r"\b((jest m|tw)oj|(tam|\b)t|wasz)ą\b\s+\bpanią\b")
edge_case_p = apply_mask(df_2nd_person, r"\b((tw|m)oim|[wn]aszym|im|(\b|tam)tym)\b\s+\bpa(ni|n)om\b")

mask_m = apply_mask(df_2nd_person, r"\bpana\b") & ~edge_case_m
mask_f = apply_mask(df_2nd_person, r"\bpanią\b") & ~edge_case_f
mask_p = apply_mask(df_2nd_person, r"\bpa(ni|n)om\b") & ~edge_case_p

print(f"Num edge cases --> [Female]: {edge_case_f.sum()} | [Male]: {edge_case_m.sum()}    | [Plural]: {edge_case_p.sum()}")
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()} | [Plural]: {mask_p.sum()}")
df_2nd_person, df_proc_f, df_proc_m, df_proc_p = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, df_proc_p, mask_f, mask_m, mask_p)

### **GROUP 2C**: Formal address **( [panu/panem/mężczyzno] / kobieto )**

In [ ]:
edge_case_m1 = apply_mask(df_2nd_person, r"\b((tw|m)ojemu|[wn]aszemu|mu|(\b|tam)temu)\b\s+\bpanu\b")
edge_case_m2 = apply_mask(df_2nd_person, r"\b(((kiedy(ś|\b)|nikt|z\b).+)moim|twoim|waszym|tym|jest)\b\s+\bpanem\b")

mask_m = (apply_mask(df_2nd_person, r"\bpanu\b") & ~edge_case_m1) | (apply_mask(df_2nd_person, r"\bpanem\b") & ~edge_case_m2)
mask_m = mask_m | apply_mask(df_2nd_person, r"\bmężczyzno\b")
mask_f = apply_mask(df_2nd_person, r"\bkobieto\b")

print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()}")

df_2nd_person, df_proc_f, df_proc_m = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, None, mask_f, mask_m, None)

### **GROUP 3A**: Plural-focused extraction - **leftover verbs**

In [ ]:
def extract_single(df_main, df_proc_x, mask_x):
    df_proc_x = pd.concat([df_proc_x, df_main[mask_x]], ignore_index=True)
    df_main = df_main[~(mask_x)].reset_index(drop=True)
    return df_main, df_proc_x

In [ ]:
mask_p = apply_mask(df_2nd_person, r"\b((((sprób|zaplan)ow|[gb]r|d|napis)a|(m|widz)ie|(rob|pros|ustaw)i|by)li|jeste|zgło|powinni)[śs]cie\b")
print(f"Num examples --> [Plural]: {mask_p.sum()}")
df_2nd_person, df_proc_p = extract_single(df_2nd_person, df_proc_p, mask_p)

### **GROUP 3B**: Plural-focused extraction - **verbs ending in the suffix -cie**

In [ ]:
verbs_cie = df_2nd_person['pol_text'].str.extract(r"\b(\w+cie)\b", flags=re.IGNORECASE)[0].str.lower().value_counts()
verbs_cie = list(verbs_cie.to_dict())
print(f"verbs_cie length: {len(verbs_cie)}")
print(verbs_cie[:13])

In [ ]:
verbs_cie = df_2nd_person['pol_text'].str.extract(r"\b(\w+cie)\b", flags=re.IGNORECASE)[0].str.lower().value_counts()
verbs_cie = list(verbs_cie.to_dict())
print(f"verbs_cie length: {len(verbs_cie)}")
print(verbs_cie[:13])

In [ ]:
kickout_re = r"\b(\w+([śkłęąuro]|([eoiu]|[^t]a)n)|\w*(([^ż]|\b)r[^azw]|(?<![csr]z)(?<![^\b]ż)y|k.[^jżrpzbi]))cie\b"
print(f"Num removed: {sum(bool(re.search(kickout_re, x)) for x in verbs_cie)}")
verbs_cie = [x for x in verbs_cie if not bool(re.search(kickout_re, x))]
print(f"verbs_cie length: {len(verbs_cie)}")

In [ ]:
with open("../local_data/gender_pronouns/2nd_person/word_lists/kickout_words.pkl", 'rb') as f:
    kickout_words = pickle.load(f)


In [ ]:
mask_p_1 = apply_mask(df_2nd_person, r"\b\w+cie\b")
mask_p_2 = apply_mask(df_2nd_person[mask_p_1], rf"\b({'|'.join(verbs_cie)})\b")
mask_p = (mask_p_1 & mask_p_2)

print(f"Num examples --> [Plural]: {mask_p.sum()}")
df_2nd_person, df_proc_p = extract_single(df_2nd_person, df_proc_p, mask_p)

### **GROUP 3C**: Plural-focused extraction  **-was / -wasz** etc.

In [ ]:
edge_case_p = apply_mask(df_2nd_person, r"\bwaszej\b\s+\bwysoko[śs]ci\b") | apply_mask(df_2nd_person, r"\bwasza\b\s+\bwysoko[śs][ćc]\b")
edge_case_p = edge_case_p | apply_mask(df_2nd_person, r"\bwasza\b\s+\bŁaskawo[śs][ćc]\b")

mask_p = apply_mask(df_2nd_person, r"\b(wa[sm](\b|i)|wasz(\b|[aeąyę]|emu|ej|ych|ym|ego|ymi))\b") & ~edge_case_p

print(f"Num examples --> [Plural]: {mask_p.sum()} | Num edge cases --> [Plural]: {edge_case_p.sum()}")
df_2nd_person, df_proc_p = extract_single(df_2nd_person, df_proc_p, mask_p)

In [ ]:
mask_p = apply_mask(df_2nd_person, r"\bwy\b")
print(f"Num examples --> [Plural]: {mask_p.sum()}")
df_2nd_person, df_proc_p = extract_single(df_2nd_person, df_proc_p, mask_p)

### **GROUP 4**: Jesteś - Manual Profession/Noun Conversion

- #### **1.** -arzem --> -arką  **( e.g. lekarzem-lekarką | pisarzem-pisarką )**

In [ ]:
def get_female_nouns(df_main, nouns_dict, suffixes_dct):
    nouns_pattern = r'|'.join(re.escape(key) for key in nouns_dict.keys())
    mask_nouns = apply_mask(df_main, rf'\bjesteś\b.+\b({nouns_pattern})\b') & ~apply_mask(df_main, r'\bjestem\b')
    print(f"Num examples: {mask_nouns.sum()}")
    female_nouns = df_main[mask_nouns].reset_index(drop=True)
    dct_comb = nouns_dict | suffixes_dct
    pattern_comb = r'|'.join(re.escape(key) for key in dct_comb.keys())
    female_nouns['pol_text'] = female_nouns['pol_text'].apply(lambda snt: re.sub(pattern_comb, 
                                                                                 lambda match: dct_comb[match.group(0).lower()], 
                                                                                 snt, flags=re.IGNORECASE))
    return female_nouns, mask_nouns

In [ ]:
nouns_male = df_2nd_person['pol_text'].str.extract(r'\bjesteś\b.+\b(\w+arzem)\b', flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = nouns_male[nouns_male >= 3].to_dict()
print(nouns_male)

In [ ]:
suffix_conversion = {'dobrym': 'dobrą', 'kim': 'ką', 'ym': 'ą', 'łeś': 'łaś', 'kiem': 'czką', 'moim': 'moją', 'tary': 'tara', 
                     'durniu': 'idiotko', 'głupcem': 'idiotką', 'czem': 'czką', 'rem': 'rką', 'tanem': 'tanką', 'grafem': 'grafką'}
nouns_1a = ['lekarzem', 'pisarzem', 'dziennikarzem', 'kucharzem', 'sekretarzem', 'malarzem', 'ochroniarzem', 'marynarzem',
            'handlarzem', 'żeglarzem', 'piosenkarzem']
nouns_1a = {m_noun: re.sub(r"rzem\b", "rką", m_noun) for m_noun in nouns_1a}
female_nouns_1a, mask_nouns_1a = get_female_nouns(df_2nd_person, nouns_1a, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_1a sample:\n\n{female_nouns_1a['pol_text'].sample(3)}\n{'='*60}")

In [ ]:
nouns_1b = {'szczęściarzem': 'szczęściarą', 'gospodarzem': 'gospodynią', 'nudziarzem': 'nudziarą', 'spryciarzem': 'spryciarą',
            'komisarzem': 'panią komisarz'}
female_nouns_1b, mask_nouns_1b = get_female_nouns(df_2nd_person, nouns_1b, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_1b sample:\n\n{female_nouns_1b['pol_text'].sample(3)}\n{'='*60}")

In [ ]:
df_proc_f = pd.concat([df_proc_f, female_nouns_1a, female_nouns_1b], ignore_index=True)
df_proc_m = pd.concat([df_proc_m, df_2nd_person[mask_nouns_1b | mask_nouns_1a]], ignore_index=True)
df_2nd_person = df_2nd_person[~(mask_nouns_1b | mask_nouns_1a)].reset_index(drop=True)

- #### **2.** -mistrzem --> -mistrzynią **( e.g. [bur]mistrzem - [bur]mistrzynią )**

In [ ]:
nouns_male = df_2nd_person['pol_text'].str.extract(r'\bjesteś\b.+\b(\w*mistrzem)\b', flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = nouns_male[nouns_male >= 3].to_dict()
print(nouns_male)

In [ ]:
nouns_2 = {'mistrzem': 'mistrzynią', 'burmistrzem': 'burmistrzynią'}
suffix_conversion = {'kim': 'ką', 'ym': 'ą', 'nikiem': 'nicą', 'moim': 'moją', 'rem': 'rką', 'nem': 'nką', 'jakiego': 'jaką',
                     'grany': 'grana'}
female_nouns_2, mask_nouns_2 = get_female_nouns(df_2nd_person, nouns_2, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_2 sample:\n\n{female_nouns_2['pol_text'].sample(3)}\n{'='*60}")

In [ ]:
df_proc_f = pd.concat([df_proc_f, female_nouns_2], ignore_index=True)
df_proc_m = pd.concat([df_proc_m, df_2nd_person[mask_nouns_2]], ignore_index=True)
df_2nd_person = df_2nd_person[~(mask_nouns_2)].reset_index(drop=True)

- #### **3.** -nikiem --> -niczką/cą **(e.g. ochotnikiem -> ochotniczką | nieudacznikiem -> nieudacznicą)**

In [ ]:
nouns_male = df_2nd_person['pol_text'].str.extract(r'\bjesteś\b.+\b(\w+ikiem)\b', flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = nouns_male[nouns_male >= 3].to_dict()
print(nouns_male)

In [ ]:
nouns_3b = ['niewolnikiem', 'anglikiem', 'pracownikiem', 'grzesznikiem', 'nieudacznikiem', 'robotnikiem']
nouns_3a = [k for k in nouns_male.keys() if k not in nouns_3b+['pułkownikiem', 'szkodnikiem']]

nouns_3b = {k: re.sub(r"nikiem\b|likiem\b", lambda x: {'nikiem': 'nicą', 'likiem': 'ielką'}[x.group(0)], k) for k in nouns_3b}
nouns_3a = {k: re.sub(r"ikiem\b", 'iczką', k) for k in nouns_3a}

In [ ]:
suffix_conversion = {'nim': 'nią', 'ym': 'ą', 'moim': 'moją', 'wielkim': 'wielką', 'jakimś': 'jakąś', 'takim': 'taką',
                     'tem': 'tką', 'łeś': 'łaś', 'jakiego': 'jaką', 'szny': 'szna'}
female_nouns_3, mask_nouns_3 = get_female_nouns(df_2nd_person, nouns_3a | nouns_3b, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_3 sample:\n\n{female_nouns_3['pol_text'].sample(3)}\n{'='*60}")

In [ ]:
df_proc_f = pd.concat([df_proc_f, female_nouns_3], ignore_index=True)
df_proc_m = pd.concat([df_proc_m, df_2nd_person[mask_nouns_3]], ignore_index=True)
df_2nd_person = df_2nd_person[~(mask_nouns_3)].reset_index(drop=True)

- #### **4.** Final **-em** Noun Conversion

In [ ]:
nouns_male = df_2nd_person['pol_text'].str.extract(r'\bjesteś\b.+\b(\w+em)\b', flags=re.IGNORECASE)[0].str.lower().value_counts()
nouns_male = list(nouns_male[nouns_male >= 5].to_dict())
print(nouns_male)

In [ ]:
with open("../local_data/gender_pronouns/2nd_person/word_lists/nouns_4a_source.pkl", 'rb') as f:
    nouns_4a = pickle.load(f)

nouns_conversion_4a = {'orem': 'orką', 'erem': 'erką', 'entem': 'entką', 'antem': 'antką', 'aninem': 'anką', 'anem': 'anką',
                       'akiem': 'aczką', 'ykiem': 'yczką', 'kiem': 'kinią', 'cielem': 'cielką', 'ejem': 'ejką',
                       'owcem': 'owczynią','atem': 'atką', 'fem': 'fką', 'aczem': 'aczką', 'ogiem': 'ożką', 'rtem': 'rtką'}

nouns_4a = {k: re.sub(rf"{r'|'.join(nouns_conversion_4a)}", lambda x: nouns_conversion_4a[x.group(0)], k) for k in nouns_4a}

In [ ]:
with open("../local_data/gender_pronouns/2nd_person/word_lists/nouns_4b_source.pkl", 'rb') as f:
    nouns_4b = pickle.load(f)

nouns_conversion_4b = {'cielem': 'ciółką', 'telem': 'telką', 'plem': 'pelką', 'fem': 'fową', 'źniem': 'źniarką', 'czniem': 'czennicą', 'nem': 'nką',
                       'wem': 'wką', 'ydem': 'ydówką', 'tem': 'tką', 'szem': 'szką', 'ńcem': 'nką', 'uzem': 'uzką', 'rzem': 'rką', 'nkiem': 'nką',
                       'łem': 'licą', 'chem': 'szką'}
nouns_4b = {k: re.sub(rf"{r'|'.join(nouns_conversion_4b)}", lambda x: nouns_conversion_4b[x.group(0)], k) for k in nouns_4b}

In [ ]:
suffix_conversion = {'nim': 'nią', 'moim': 'moją', 'wielkim': 'wielką', 'jakimś': 'jakąś', 'takim': 'taką', 'tem': 'tką',
                     'łeś': 'łaś', 'jakiego': 'jaką', 'szny': 'szna', 'nym': 'ną', 'tym': 'tą', 'rym': 'rą', 'wym': 'wą',
                     'łym': 'łą', 'szym': 'szą', 'cym': 'cą', 'skim': 'ską', 'ckim': 'cką', 'erem': 'erką', 'tystą': 'tystką',
                     'istą': 'istką', 'głupcem': 'idiotką', 'niemcem': 'niemką'}
female_nouns_4, mask_nouns_4 = get_female_nouns(df_2nd_person, nouns_4a | nouns_4b, suffix_conversion)
print(f"{'='*60}\nfemale_nouns_4 sample:\n\n{female_nouns_4['pol_text'].sample(3)}\n{'='*60}")

In [ ]:
df_proc_f = pd.concat([df_proc_f, female_nouns_4], ignore_index=True)
df_proc_m = pd.concat([df_proc_m, df_2nd_person[mask_nouns_4]], ignore_index=True)
df_2nd_person = df_2nd_person[~(mask_nouns_4)].reset_index(drop=True)

### **GROUP 4A**: [Jesteś, Będziesz, -byś, ...być] --> **( [-ła, -ą] / [-ł, -ły, -ym] ) cases**

In [ ]:
def get_spec_pattern(df_main, re_patt, f_name):
    mask = apply_mask(df_main, re_patt)
    main_mask = df_main[mask]
    suffix_words = set(df_main['pol_text'].str.extract(re_patt, flags=re.IGNORECASE)[0].str.lower().value_counts().to_dict())
        
    with open(f"../local_data/gender_pronouns/2nd_person/word_lists/{f_name}.pkl", 'rb') as f:
        mismatch_list = pickle.load(f)
            
    for word in mismatch_list:
        suffix_words.discard(word)
    return mask & apply_mask(main_mask, rf"\b({'|'.join(suffix_words)})\b")

In [ ]:
pat_m = r"\bjesteś\b.+\b(([cm]|(b|wspan|(\b|nie)śm|(za|wy)rozum|zgorzkn|skretyn|odrętw|oniem)i|(\b|nie)doskon|(\b|nie)dojrz|\
[oz]bol|[oz]bol|wytrzym|(nieby|zuch|wytr)w|niedojrz|(\b|nie)st)a|(\b|nie)(mi|czu|zwyk)|wściek|niez|doros|pod|(prze|\b)bieg|weso|\
weso|szczup|go|odleg|by|uleg|ciep|węz|ozięb|rozwiąz|wąt|umar|z|stetrycza)ły\b"
pat_f = r"\bjesteś\b.+\b(z|m[ia]|(c|doskon|wspani|(wy|za)rozumi|bi|(nie|\b)śmi|wytrzym|zgorzkni|spleśni)a|pod|(\b|nie)(czu|mi)|niez|ciep|osch|\
doros|doros|wściek|niezwyk|szczup|zgni|rozwiąz|weso|zadręcza|odleg|szczup|by|go|(\b|nie)dojrza|(prze|\b)bieg|ozięb|oszala)ła\b"

mask_m = apply_mask(df_2nd_person, pat_m) | apply_mask(df_2nd_person, r"\b(\w*byś|b[ęe]dziesz|\w+(esz|esteś|isz|cznij)\b.+\bbyć)\b.+\b\w+([^ó]ł|ły)\b")
mask_f = apply_mask(df_2nd_person, pat_f) | apply_mask(df_2nd_person, r"\b(\w*byś|b[ęe]dziesz|\w+(esz|esteś|isz|cznij)\b.+\bbyć)\b.+\b\w+ła\b")
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()}")
df_2nd_person, df_proc_f, df_proc_m = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, None, mask_f, mask_m, None)

In [ ]:
pat_m = r"\b(jesteś|b[ęe]dziesz|\w+(esz|esteś|isz|cznij)\b.+\bbyć)\b.+\b((potwor|chaos|człowieki|przypadki)em|kimś|\
|humorze|w tym\b\s+\b\w+[^y]\b|domowym|kim jesteś|z każdym)\b"
pat_f = r"\b(jesteś|b[ęe]dziesz|\w+(esz|esteś|isz|cznij)\b.+\bbyć)\b.+\b((osob|wolności|rybk|świni|kas|kaczk|małpk|rzecz|\
|zagadk|kreatur|istot|twarz|nagrod|nadziej|maszyn)ą|\w+(nien|nym)|kasę)\b"

ignore_m = apply_mask(df_2nd_person, pat_m)
ignore_f = apply_mask(df_2nd_person, pat_f)

mask_m = (get_spec_pattern(df_2nd_person, r"\bjesteś\b.+\b(\w+ym)\b", 'mismatch_suffix_ym') | 
          apply_mask(df_2nd_person, r"\b(b[ęe]dziesz|\w+(esz|esteś|isz|cznij)\b.+\bbyć)\b.+\b\w+ym\b")) & ~ignore_m

mask_f = (get_spec_pattern(df_2nd_person, r"\bjesteś\b.+\b(\w+ą)\b", 'mismatch_suffix_a') | 
          apply_mask(df_2nd_person, r"\b(b[ęe]dziesz|\w+(esz|esteś|isz|cznij)\b.+\bbyć)\b.+\b\w+ą\b")) & ~ignore_f

print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()}")
df_2nd_person, df_proc_f, df_proc_m = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, None, mask_f, mask_m, None)

### **GROUP 4B**: [Jesteś, Będziesz] --> specific **[-ta, -na, -ra, -wa, -ka] / [-ty, -ny, -ry, -wy, -im, -ki]**

In [ ]:
def apply_list_spec(spec_suffixes, mask_init):
    for suf, f_suf in spec_suffixes.items():
        mask_suffix = get_spec_pattern(df_2nd_person, rf'\bjesteś\b.+\b(\w+{suf})\b', f'mismatch_suffix_{f_suf}')
        mask_init = mask_init | mask_suffix
    return mask_init

In [ ]:
spec_suffixes_f = {'ta': 'ta', 'na': 'na', 'ra': 'ra', 'wa': 'wa', 'ka': 'ka', 'ca': 'ca'}
spec_suffixes_m = {'ty': 'ty', 'ny': 'ny', 'ry': 'ry', 'wy': 'wy', 'im': 'im', 'ki': 'ki', 'cy': 'cy'}

ignore_f = apply_mask(df_2nd_person, r"\bjesteś\b.+\b((osob|wolności|rybk|świni|kas|kaczk|rzecz|zagadk|kreatur|istot|twarz|nagrod|nadziej)ą)\b")
ignore_m = apply_mask(df_2nd_person, r"\bjesteś\b.+\b((potwor|chaos|człowieki|przypadki)em|domowym)\b")

patt_m = r'\bjesteś\b.+\b((głu|ci)chy|(mło|twar|bla|chu)dy|((\b|nie)uprzej|świado|nie|niewido)my|(sła|gru)by|sam|winien|głupi|\
           |(ostat|bezpośred|odpowied|((\b|nie)pełno|nie)let)ni|(uro|spostrzegaw|poryw|stanow|nadopiekuń|(tajem|zasad)ni)czy|mój|\
           |pewien|godzie[nń]|(śle|tę|ską)py)\b'
final_cases_m = apply_mask(df_2nd_person, patt_m)

mask_m = (apply_list_spec(spec_suffixes_m, apply_mask(df_2nd_person, r'\bjesteś\b.+\b\w+szy\b')) | final_cases_m) & ~ignore_m

patt_f = r'\bjesteś\b.+\b(((głu|li|ci|su|kłamczu)ch|((\b|nie)uprzej|świado|ma)m|(mło|bla|twar|chu)d|(sła|gru|ba)b|sam|głupi|\
           |((\b|nie)pełnolet|nielet|ostat|odpowied|bezpośred)ni|((\b|prze)uro|spostrzegaw|stanow|(tajem|stron|zasad)ni)cz|moj)a|\
           |\w+czko|(śle|tę)pa)\b'
final_cases_f = apply_mask(df_2nd_person, patt_f)
mask_f = (apply_list_spec(spec_suffixes_f, apply_mask(df_2nd_person, r'\bjesteś\b.+\b\w+sza\b')) | final_cases_f) & ~ignore_f
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()}")
df_2nd_person, df_proc_f, df_proc_m = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, None, mask_f, mask_m, None)

In [ ]:
mask_f = apply_mask(df_2nd_person, r"\bb[ęe]dziesz\b.+\b(sama|\w+[wkn]a)\b")
mask_m = apply_mask(df_2nd_person, r"\bb[ęe]dziesz\b.+\b(sam|\w+[wkn]y)\b")
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()}")
df_2nd_person, df_proc_f, df_proc_m = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, None, mask_f, mask_m, None)

### **GROUP 4C**: Jesteś --> **-em** / Male: **-ą**

In [ ]:
with open(f"../local_data/gender_pronouns/2nd_person/word_lists/universal_words_em.pkl", 'rb') as f:
    universal_em = pickle.load(f)
    
mask_suffix_em = apply_mask(df_2nd_person, r'\bjesteś\b.+\b\w+em\b')
mask_f = mask_suffix_em & apply_mask(df_2nd_person[mask_suffix_em],  rf"\bjesteś\b.+\b({'|'.join(universal_em)})\b")
mask_m = get_spec_pattern(df_2nd_person, r'\bjesteś\b.+\b(\w+em)\b', 'mismatch_suffix_em')
print(f"Num examples --> [Female]: {mask_f.sum()} | [Male]: {mask_m.sum()}")
df_2nd_person, df_proc_f, df_proc_m = extract_pronouns(df_2nd_person, df_proc_f, df_proc_m, None, mask_f, mask_m, None)

In [ ]:
with open(f"../local_data/gender_pronouns/2nd_person/word_lists/male_suffix_a.pkl", 'rb') as f:
    male_suffix_a = pickle.load(f)

mask_suffix_a = apply_mask(df_2nd_person, r'\bjesteś\b.+\b\w+ą\b')
mask_m = mask_suffix_a & apply_mask(df_2nd_person[mask_suffix_a],  rf"\bjesteś\b.+\b({'|'.join(male_suffix_a)})\b")
print(f"Num examples --> [Male]: {mask_m.sum()}")
df_2nd_person, df_proc_m = extract_single(df_2nd_person, df_proc_m, mask_m)

## **5.** Saving The Data

In [ ]:
df_text = df_text[~apply_mask(df_text, r"\b\w+(ł[ea]ś|[^s]łabym|łbym)\b")].reset_index(drop=True)

In [ ]:
df_first_person = df_first_person[~apply_mask(df_first_person, r"\b\w+ł[ea]ś\b")].reset_index(drop=True)

In [ ]:
df_proc_m['addr_ref'] = 'M'
df_proc_f['addr_ref'] = 'F'
df_proc_p['addr_ref'] = 'P'
df_2nd_person['addr_ref'] = 'NA'

In [ ]:
df_male['self_ref'] = 'M'
df_male['addr_ref'] = 'NA_OTHER'

df_female['self_ref'] = 'F'
df_female['addr_ref'] = 'NA_OTHER'

df_first_person['self_ref'] = 'NA'
df_first_person['addr_ref'] = 'NA_OTHER'

df_text['self_ref'] = 'NA_OTHER'
df_text['addr_ref'] = 'NA_OTHER'

In [ ]:
final_data = pd.concat([df_proc_m, df_proc_f, df_proc_p, df_2nd_person, df_male, df_female, df_first_person, df_text], ignore_index=True)

In [ ]:
report_shapes(proc_m=df_proc_m, proc_f=df_proc_f, proc_p=df_proc_p, second_person_other=df_2nd_person,
              male=df_male, female=df_female, first_person=df_first_person, text=df_text, final=final_data)

In [ ]:
save_dir = "../local_data/gender_pronouns/2nd_person"
os.makedirs(save_dir, exist_ok=True)

with open(f"{save_dir}/data_final_2nd_person.pkl", 'wb') as f:
    pickle.dump(final_data, f)